In [ ]:
import pandas as pd
from utils import generate_random_color, first

In [ ]:
data_folder = 'toy_data_3'

In [ ]:
rinc	= pd.read_csv(f'data/{data_folder}/real_income.csv',	sep=r'\s*,\s*',index_col=False)
rexps	= pd.read_csv(f'data/{data_folder}/real_expenses.csv',	sep=r'\s*,\s*',index_col=False)
vinc	= pd.read_csv(f'data/{data_folder}/virtual_income.csv',	sep=r'\s*,\s*',index_col=False)
vexps	= pd.read_csv(f'data/{data_folder}/virtual_expenses.csv',	sep=r'\s*,\s*',index_col=False)

# Generate Cashflow Tables

In [ ]:
income_cashflow = pd.merge(rinc,vinc,how='left',on='txid').rename(columns = {
	'from_x': 'from',
	'to_x': 'via',
	'to_y': 'to',
	'amount_y': 'amount'
})[ ['txid','vtxid','from', 'via','to','amount'] ] # reorder so that txid and vtxid are next to each other, and we leave out what we drop in expense cf calculation below
income_cashflow.head(5)

In [ ]:
vexps

In [ ]:
# The way that the day is laid out, the way to generate a cashflow for expenses differs from the above for income chart
# so I will just do both of them manually
expenses_cashflow = pd.merge(vexps,rexps,how='left',on='txid').rename(columns={
	'from_x': 'from',
	'to_y': 'to',
	'from_y': 'via',
	'amount_y': 'amount'
}).drop(columns=['to_x', 'amount_x'])
expenses_cashflow.head(5)

# Mine Data

## Get Aggregate

In [ ]:
income_amount_by_source = income_cashflow.groupby([ 'from', 'via' ]).agg({'amount': 'sum'})
income_amount_by_source.head(5)

In [ ]:
inc_cashflow = income_cashflow.groupby(['from','via','to']).agg({'amount': 'sum'})
inc_cashflow .head(5)

In [ ]:
vacc_to_raccouts = expenses_cashflow.groupby(['from', 'via']).agg({'amount': 'sum'})
vacc_to_raccouts.head(5)

In [ ]:
out_cashflow_amts = expenses_cashflow.groupby(['from','via','to']).agg({'amount': 'sum'})
vacc_to_raccouts.head(5)

## Color Tables

In [ ]:
inc_accs = set(rinc['from'])

inc_colors = { ri: generate_random_color() for ri in inc_accs }
inc_colors

In [ ]:
vaccs = map(first,vacc_to_raccouts.index)
vacc_colors = { vacc: generate_random_color() for vacc in vaccs}
vacc_colors

# Build Visualizer

In [ ]:
import visualizer as vv

## Add Layers

In [ ]:
import setup_visualize as setup

########################
# Add Layers          #
#######################
vis = vv.LayerVisualizer()
ri  = vis.add_layer('Real Income')
ra  = vis.add_layer('Real Accounts Income')
va  = vis.add_layer('Virtual Accounts')
raa = vis.add_layer('Real Accounts Outcome')
ro  = vis.add_layer('Real Outflow')

########################
# Add Accounts         #
########################

# Real Income
setup.add_rinc_nodes(ri, rinc)
# Real Accounts
setup.add_racc_nodes(ra, rinc)
# Virtual Accounts     
"""
SETUP CUSTOM_VACCS
If you want to add custom virtual accounts do
  > setup.add_vacc_nodes(va, vaccs=['VA1', 'VA2', ..., 'VAN])
"""
setup.add_vacc_nodes(va)
# Real Accounts (Reflow from Virtual Accounts)
setup.add_raccout_nodes(raa,rexps)
# Real Outflow
setup.add_rout_nodes(ro, rexps)

#######################
# Add Connections     #
#######################

setup.link_rincs_to_raccs(
        ri, ra, 
        income_amount_by_source, inc_colors
)
setup.link_raccs_to_vaccs(
        ra, va, 
        inc_cashflow, inc_colors
)

setup.link_vaccs_to_raccouts(
        va, raa, 
        vacc_to_raccouts, vacc_colors
)
setup.link_raccouts_to_routs(
        raa, ro, 
        out_cashflow_amts, vacc_colors
)

vis.show()